DBFS storage not useable by service principle and is external so not under governance so we are trying to use volume instead and clean this temp stoage ourselves.

**Once there are 3 dbx this logic will need some change, for the poc the catalog is simulating different databricks**

In [0]:
%sql
-- Dev environment
CREATE SCHEMA IF NOT EXISTS dev_catalog.temp;

CREATE VOLUME IF NOT EXISTS dev_catalog.temp.integration_testing
COMMENT 'Temporary storage for integration tests - auto-cleaned by test fixtures';

-- Staging environment
CREATE SCHEMA IF NOT EXISTS staging_catalog.temp;

CREATE VOLUME IF NOT EXISTS staging_catalog.temp.integration_testing
COMMENT 'Temporary storage for integration tests - auto-cleaned by test fixtures';


Allow principles to use temp and users to view it

In [0]:
%skip
# ai test
import uuid
from datetime import datetime
import pytz

# -------------------------------
# VARIABLES
# -------------------------------

# Choose environment
catalog = "dev_catalog"        # Change to "staging_catalog" to test staging
volume_name = "integration_testing"

# Timezone for readable timestamp
tz = pytz.timezone("UTC")      # Change to your timezone if desired
timestamp = datetime.now(tz).strftime("%Y%m%d_%H%M%S")

# -------------------------------
# PER-RUN DIRECTORY
# -------------------------------

run_id = f"{timestamp}_{uuid.uuid4()}"  # timestamp + uuid for uniqueness
run_path = f"/Volumes/{catalog}/temp/{volume_name}/run_{run_id}"

print(f"Creating per-test run directory:\n{run_path}")
dbutils.fs.mkdirs(run_path)
print("✅ Directory created successfully.")

# -------------------------------
# TEST FILE
# -------------------------------

test_file = f"{run_path}/hello.txt"
test_content = "integration test works"

print(f"Writing test file:\n{test_file}\nContent: {test_content}")
dbutils.fs.put(test_file, test_content, overwrite=True)
print("✅ File written successfully.")

# -------------------------------
# READ BACK
# -------------------------------

content = dbutils.fs.head(test_file)
print(f"Reading back file:\n{test_file}\nContent read: {content}")

assert content == test_content, "File content does not match expected value!"
print("✅ File content matches expected value.")




In [0]:
%skip
# ai cleanup
# -------------------------------
# OPTIONAL CLEANUP
# -------------------------------

# Uncomment to remove the run folder after testing
dbutils.fs.rm(run_path, recurse=True)
print(f"✅ Run directory cleaned up: {run_path}")

